# Qwen 1.5B Notebook

Lucas Harrich

This notebook takes the prompts from `dataset_clean.csv`, runs them through Qwen 1.5B, and saves the results in `submission_qwen_1_5b.csv`.

You can use it with a small sample first or run it on the full dataset.

Short model info:
- Model: `Qwen/Qwen2.5-1.5B-Instruct`
- Size: about `1.5B` parameters
- Input length here: `420` tokens
- Output length here: up to `160` new tokens


In [ ]:
%pip install -q torch transformers pandas ipython


In [ ]:
# Imports and basic settings
from pathlib import Path as pathClass
import os
import pandas as pandasLib
import torch
from IPython.display import display as showTable
from transformers import AutoTokenizer as autoTokenizerClass
from transformers import AutoModelForCausalLM as autoModelForCausalLmClass

# Input and output files
inputPath = pathClass("dataset_clean.csv")
outputPath = pathClass("inference_qwen_1_5b.csv")

# Lightweight run settings
useSample = True
sampleSize = 5
modelName = "Qwen/Qwen2.5-1.5B-Instruct"
maxInputTokens = 420
maxNewTokens = 160

# Instruction prefix for every prompt
systemPrompt = (
    "Beantworte die oesterreichische Steuerfrage auf Deutsch. "
    "Sei kurz. Erfinde keine Fakten."
)

# Required input schema
requiredColumns = {"id", "prompt"}


In [ ]:
# Load and validate the input data
sourceTable = pandasLib.read_csv(inputPath, encoding="utf-8-sig")

missingColumns = requiredColumns - set(sourceTable.columns)
if missingColumns:
    raise ValueError(f"Missing columns: {sorted(missingColumns)}")

sourceTable["prompt"] = sourceTable["prompt"].fillna("").astype(str)

workingTable = sourceTable.head(sampleSize).copy() if useSample else sourceTable.copy()
print(f"rows to process: {len(workingTable)}")
showTable(workingTable.head())


In [ ]:
# Pick the best available device
def pickDevice():
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

# Load tokenizer and model once
def loadGeneratorBundle():
    activeDevice = pickDevice()

    if activeDevice.type == "cpu":
        torch.set_num_threads(max(1, (os.cpu_count() or 2) - 1))

    activeTokenizer = autoTokenizerClass.from_pretrained(
        modelName,
        use_fast=True
    )
    if activeTokenizer.pad_token is None:
        activeTokenizer.pad_token = activeTokenizer.eos_token

    activeModel = autoModelForCausalLmClass.from_pretrained(modelName)
    activeModel.to(activeDevice)
    activeModel.eval()
    return activeTokenizer, activeModel, activeDevice

# Normalize text before and after generation
def cleanText(value):
    return " ".join(str(value).strip().split())

# Build the final model prompt
def buildPromptText(sourcePrompt):
    promptText = cleanText(sourcePrompt)
    if not promptText:
        return ""

    if getattr(tokenizer, "chat_template", None):
        messages = [
            {"role": "system", "content": systemPrompt},
            {"role": "user", "content": promptText}
        ]
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

    promptLines = [
        f"Instruction: {systemPrompt}",
        f"Question: {promptText}",
        "Answer:"
    ]
    return "\n".join(promptLines)

# Run generation for a single prompt
def generateAnswer(sourcePrompt):
    fullPrompt = buildPromptText(sourcePrompt)
    if not fullPrompt:
        return ""

    tokenBatch = tokenizer(
        fullPrompt,
        return_tensors="pt",
        truncation=True,
        max_length=maxInputTokens
    )
    tokenBatch = {name: value.to(device) for name, value in tokenBatch.items()}

    with torch.inference_mode():
        generatedIds = model.generate(
            **tokenBatch,
            max_new_tokens=maxNewTokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.05
        )

    answerIds = generatedIds[0][tokenBatch["input_ids"].shape[1]:]
    answerText = tokenizer.decode(answerIds, skip_special_tokens=True)
    return cleanText(answerText)

# Initialize the generator
tokenizer, model, device = loadGeneratorBundle()
print(f"model: {modelName}")
print(f"device: {device}")


In [ ]:
# Generate answers and save the output CSV
answerList = []

for rowIndex, row in enumerate(workingTable.itertuples(index=False), start=1):
    answerText = generateAnswer(row.prompt)
    answerList.append({"id": row.id, "answer": answerText})
    print(f"{rowIndex}/{len(workingTable)}: {row.id}")

resultTable = pandasLib.DataFrame(answerList, columns=["id", "answer"])
resultTable.to_csv(outputPath, index=False, encoding="utf-8")

print(f"saved: {outputPath.resolve()}")
showTable(resultTable.head())
